#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
# RENAME_MAP is a dictionary that maps original column names in the DataFrame
# to their new names for easier readability and consistency.
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data Transformations

## Trimming string fields

In [0]:
# Iterate over all fields in the DataFrame schema
for field in df.schema.fields:
    # Check if the field is of StringType
    if isinstance(field.dataType, StringType):
        # Trim leading and trailing whitespace from string columns
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalizations

In [0]:
df = (
    # Normalize 'cst_marital_status' column: map 'S' to 'Single', 'M' to 'Married', else 'n/a'
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    # Normalize 'cst_gndr' column: map 'F' to 'Female', 'M' to 'Male', else 'n/a'
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

#Renaming the columns

In [0]:
# This code iterates through the RENAME_MAP dictionary, which contains mappings of old column names to new column names.
# For each pair, it renames the corresponding column in the DataFrame 'df' to its new name for improved readability and consistency.
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Write Into Silver Table

In [0]:
# Write the DataFrame 'df' into the Delta table 'silver.crm_customers' in overwrite mode.
(
    df.write
     .mode("overwrite")  # Overwrites the existing table data. Good for small tables.
     .format("delta")    # Specifies Delta Lake format for storage.
     .saveAsTable("silver.crm_customers")  # Saves the DataFrame as a managed table.
)